In [1]:
import os

import numpy as np
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

import torch
import torchvision
from torchvision.transforms import transforms

from PIL import Image

from preprocessing_module import find_class_names_filenames, stratified_split_data_paths, NatureCityScenesDataset

In [2]:
import os
import sys
import random
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from torchvision import transforms as T

from sklearn.model_selection import train_test_split


# ---- Project paths ----
PROJECT_ROOT = Path("quadrant_dots_project").resolve()
DATA_ROOT = PROJECT_ROOT / ".." / ".." / "Datasets" / "quadrant_dots_rgb"

# Make project importable (so we can import our dataset + generator modules)
sys.path.append(str(PROJECT_ROOT))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)


Device: cpu


In [7]:
DATASET_PATH = r"./mandatory1_data"

class_names, class_filenames = find_class_names_filenames(DATASET_PATH)
x_train_paths, x_val_paths, x_test_paths, y_train, y_val, y_test = stratified_split_data_paths(DATASET_PATH, class_names, class_filenames)

transform = transforms.Compose([
    transforms.Resize((224, 224)), # AlexNet standard size: 224x224
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_set = NatureCityScenesDataset(x_train_paths, y_train, transform=transform)
val_set = NatureCityScenesDataset(x_val_paths, y_val, transform=transform)
test_set = NatureCityScenesDataset(x_test_paths, y_test, transform=transform)

BATCH_SIZE = 32
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_load = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)




subset_indices = list(range(100))
train_subset = torch.utils.data.Subset(train_set, subset_indices)
val_subset = torch.utils.data.Subset(val_set, subset_indices)

train_loader = torch.utils.data.DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = torch.utils.data.DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

In [4]:
from dataclasses import dataclass
from typing import Callable, Dict, Optional, Tuple

@dataclass
class EpochStats:
    loss: float
    acc: float


class Trainer:
    def __init__(
        self,
        model: nn.Module,
        criterion: nn.Module,
        optimizer: torch.optim.Optimizer,
        device: torch.device,
        augment_fn: Optional[Callable[[torch.Tensor], torch.Tensor]] = None,
    ):
        self.model = model
        self.criterion = criterion
        self.optimizer = optimizer
        self.device = device
        self.augment_fn = augment_fn

        self.history: Dict[str, list] = {
            "train_loss": [],
            "train_acc": [],
            "val_loss": [],
            "val_acc": [],
        }

    @staticmethod
    def _accuracy(logits: torch.Tensor, targets: torch.Tensor) -> float:
        preds = logits.argmax(dim=1)
        return (preds == targets).float().mean().item()

    def train_one_epoch(self, loader: DataLoader) -> EpochStats:
        self.model.train()

        total_loss = 0.0
        total_acc = 0.0
        n_batches = 0

        for images, targets in loader:
            images = images.to(self.device, non_blocking=True)
            targets = targets.to(self.device, non_blocking=True)

            # Apply augmentation policy only during training (optional)
            if self.augment_fn is not None:
                images = self.augment_fn(images)

            logits = self.model(images)
            loss = self.criterion(logits, targets)

            self.optimizer.zero_grad(set_to_none=True)
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()
            total_acc += self._accuracy(logits, targets)
            n_batches += 1

        return EpochStats(loss=total_loss / n_batches, acc=total_acc / n_batches)

    @torch.no_grad()
    def evaluate(self, loader: DataLoader) -> EpochStats:
        self.model.eval()

        total_loss = 0.0
        total_acc = 0.0
        n_batches = 0

        for images, targets in loader:
            images = images.to(self.device, non_blocking=True)
            targets = targets.to(self.device, non_blocking=True)

            logits = self.model(images)
            loss = self.criterion(logits, targets)

            total_loss += loss.item()
            total_acc += self._accuracy(logits, targets)
            n_batches += 1

        return EpochStats(loss=total_loss / n_batches, acc=total_acc / n_batches)

    def fit(self, train_loader: DataLoader, val_loader: DataLoader, epochs: int = 10):
        for epoch in range(1, epochs + 1):
            train_stats = self.train_one_epoch(train_loader)
            val_stats = self.evaluate(val_loader)

            self.history["train_loss"].append(train_stats.loss)
            self.history["train_acc"].append(train_stats.acc)
            self.history["val_loss"].append(val_stats.loss)
            self.history["val_acc"].append(val_stats.acc)

            print(
                f"Epoch {epoch:02d} | "
                f"train loss {train_stats.loss:.4f}, acc {train_stats.acc:.3f} | "
                f"val loss {val_stats.loss:.4f}, acc {val_stats.acc:.3f}"
            )


In [5]:
def batch_noise_augment(images: torch.Tensor, std: float = 0.03) -> torch.Tensor:
    noise = torch.randn_like(images) * std
    return torch.clamp(images + noise, 0.0, 1.0)

In [8]:
from ResNet import ResNet

model_ResNet18 = ResNet(img_channels=3, num_layers=18, num_classes=6).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_ResNet18.parameters(), lr=1e-3)

trainer = Trainer(
    model=model_ResNet18,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    augment_fn=lambda x: batch_noise_augment(x, std=0.02),
)

trainer.fit(train_loader, val_loader, epochs=8)


Epoch 01 | train loss 2.6011, acc 0.227 | val loss 1.8659, acc 0.188
Epoch 02 | train loss 1.7789, acc 0.266 | val loss 7.0213, acc 0.234
Epoch 03 | train loss 1.5114, acc 0.359 | val loss 39.6456, acc 0.172
Epoch 04 | train loss 1.2036, acc 0.430 | val loss 43.2377, acc 0.109
Epoch 05 | train loss 1.1577, acc 0.477 | val loss 14.1965, acc 0.180
Epoch 06 | train loss 1.3115, acc 0.453 | val loss 6.4939, acc 0.242
Epoch 07 | train loss 1.0102, acc 0.641 | val loss 7.4316, acc 0.102
Epoch 08 | train loss 1.4455, acc 0.516 | val loss 10.7591, acc 0.086
